In [1]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
# Ensure we have the absolute latest code (including import-abo command)
!git pull
!pip install -e ".[dev,vlm,gen]" -q

## 1. Import ABO Dataset
Make sure you have added the `khyeh0719/amazon-berkeley-objects-small` dataset to this notebook.

In [ ]:
!python -m tiger.cli import-abo \
    --listings-dir /kaggle/input/amazon-berkeley-objects-small/listings/metadata \
    --images-csv /kaggle/input/amazon-berkeley-objects-small/images/metadata/images.csv.gz \
    --images-dir /kaggle/input/amazon-berkeley-objects-small/images/small

## 2. Calibrate on ABO
This fits the new similarity thresholds for the non-fashion domain.

In [ ]:
!python -m tiger.cli calibrate

## 3. Retrain Arbiter
This retrains the Logistic Regression router on the new ABO-domain noise patterns.

In [ ]:
!python -m tiger.cli train-arbiter

## 4. Run Repair Ablation
This evaluates the repair pipeline using the Independent Verifier (SigLIP) and Generative Fallback.

In [ ]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

## 5. View Results
Compare this table with the Fashion results in your paper.

In [ ]:
import pandas as pd
import glob
# Dynamically find whichever ablation CSV was produced
csvs = sorted(glob.glob('data/outputs/repair_ablations_summary*.csv'))
print('Found CSVs:', csvs)
if csvs:
    df = pd.read_csv(csvs[-1])
    print(df.to_string())
else:
    print('No results CSV found. Check that ablate-repair ran successfully.')